In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from neo4j import GraphDatabase


In [2]:
raw_directory = '../data/raw'

In [3]:
euroscivoc = pd.read_excel(raw_directory + "/euroSciVoc" + ".xlsx")
legalbasis = pd.read_excel(raw_directory + "/legalBasis" + ".xlsx")
organization = pd.read_excel(raw_directory + "/organization" + ".xlsx")
project = pd.read_excel(raw_directory + "/project" + ".xlsx")
deliverables = pd.read_excel(raw_directory + "/projectDeliverables" + ".xlsx")
publications = pd.read_excel(raw_directory + "/projectPublications" + ".xlsx")
reports = pd.read_excel(raw_directory + "/reportSummaries" + ".xlsx")
topics = pd.read_excel(raw_directory + "/topics" + ".xlsx")
weblink = pd.read_excel(raw_directory + "/webLink" + ".xlsx")

/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/open

In [5]:
# Extract latitude and longitude from geolocation column
# Assumes geolocation is a string like 'lat,lon' or a tuple/list

def extract_lat_lon(geo):
    if pd.isnull(geo):
        return pd.Series({'latitude': None, 'longitude': None})
    if isinstance(geo, str):
        try:
            lat, lon = map(float, geo.split(','))
            return pd.Series({'latitude': lat, 'longitude': lon})
        except Exception:
            return pd.Series({'latitude': None, 'longitude': None})
    if isinstance(geo, (list, tuple)) and len(geo) == 2:
        return pd.Series({'latitude': geo[0], 'longitude': geo[1]})
    return pd.Series({'latitude': None, 'longitude': None})

organization[['latitude', 'longitude']] = organization['geolocation'].apply(extract_lat_lon)

# Select relevant columns and aggregate ecContribution
org_cols = ['organisationID', 'name', 'country', 'latitude', 'longitude', 'ecContribution']
organization_nodes = organization[org_cols].copy()
organization_nodes = organization_nodes.groupby(['organisationID', 'name', 'country', 'latitude', 'longitude'], as_index=False).agg({'ecContribution': 'sum'})
organization_nodes.rename(columns={'ecContribution': 'total_ecContribution'}, inplace=True)
organization_nodes.head()

,organisationID,name,country,latitude,longitude,total_ecContribution
0,875081233,Nyfors Teknologi AB,SE,59.284136,18.000635,0.0
1,875723179,SYNTHELIA ORGANICS SL,ES,40.547370,-3.626059,251971.2
2,875811837,Capacitor Metals Corp.,CA,49.261866,-122.955551,178500.0
3,875960053,GAC INNOVATION EAST EUROPE SRL,RO,44.436141,26.102720,0.0
4,876140182,Building between bridges,BE,50.973329,3.087657,0.0


In [17]:
def extract_topics_from_path(df, path_col, id_col):
    # Remove leading/trailing slashes, split by '/', and explode
    df = df[[id_col, path_col]].copy()
    df[path_col] = df[path_col].fillna('').apply(lambda x: x.strip('/'))
    df['topic'] = df[path_col].apply(lambda x: x.split('/') if x else [])
    df = df.explode('topic')
    df = df[[id_col, 'topic']]
    df = df[df['topic'].str.strip() != '']
    df['topic'] = df['topic'].str.strip()
    return df.reset_index(drop=True)

project_topics = extract_topics_from_path(euroscivoc, 'euroSciVocPath', 'projectID')
project_topics.rename(columns={'projectID': 'project_id'}, inplace=True)
project_topics.head(10)

,project_id,topic
0,101116741,social sciences
1,101116741,political sciences
2,101116741,government systems
3,101163161,agricultural sciences
4,101163161,"agriculture, forestry, and fisheries"
5,101163161,agriculture
6,101163161,grains and oilseeds
7,101163161,natural sciences
8,101163161,physical sciences
9,101163161,optics


In [18]:
# Merge project_topics with organization to get organization info for each project-topic
# Assumes organization has columns: organisationID, name, country, latitude, longitude, ecContribution, projectID

# First, join project_topics with organization on project_id/projectID
merged = project_topics.merge(organization, left_on='project_id', right_on='projectID', how='left')

# Group by topic and organization, aggregate number of projects and total ecContribution
agg = (
    merged.groupby([
        'topic', 'organisationID', 'name', 'country', 'latitude', 'longitude'
    ], as_index=False)
    .agg(numofProjects=('project_id', 'nunique'), totalecContribution=('ecContribution', 'sum'))
)

# Rename columns for clarity
agg.rename(columns={'name': 'organizationName'}, inplace=True)

# Show the result
agg.head(10)

,topic,organisationID,organizationName,country,latitude,longitude,numofProjects,totalecContribution
0,4G,947553425,CELERWAY COMMUNICATION AS,NO,59.895705,10.627547,1,2439500.00
1,4G,999874643,UNIVERSITETET I TROMSOE - NORGES ARKTISKE UNIV...,NO,69.679652,18.970928,1,226751.04
2,4G,999898311,UNIVERSIDAD DE MALAGA,ES,36.721086,-4.422002,1,165312.96
3,5G,877277701,STARION ESPANA S.L.,ES,40.524832,-3.771563,1,0.00
4,5G,879241078,DIGYONE GMBH,DE,48.793309,10.113844,1,200000.00
5,5G,879312470,AGRIROBOT APS,DK,55.783613,12.513566,1,210052.50
6,5G,881264692,BSPOKE SOLUTIONS PRIVATE COMPANY,EL,40.640317,22.935272,1,223500.00
7,5G,881338703,SPHYNX TECHNOLOGY SOLUTIONS AG,CH,47.174852,8.511260,1,0.00
8,5G,885831258,FUNDATIA ORANGE,RO,44.436141,26.102720,1,275000.00
9,5G,885958716,AKHETONICS GMBH,DE,52.486743,13.355505,1,567550.00


In [19]:
# Save the aggregated results to CSV
agg.to_csv('../data/processed/org_by_research.csv', index=False)
print("Saved results to '../data/processed/org_by_research.csv'")

Saved results to '../data/processed/org_by_research.csv'
